# Milestone 4

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from peft import get_peft_model, LoraConfig, TaskType
from torch.utils.data import Dataset

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
LABELS = ['A','B','C','D','E']
label2idx = {l:i for i,l in enumerate(LABELS)}

## Q1: Label Encoding

In [ ]:
train['label'] = train['answer'].map(label2idx)
print('Encoded label at index 150:', train.iloc[150]['label'])  # 2

## Q2: Prompt-Option Formatting

In [ ]:
row0 = train.iloc[0]
formatted = str(row0['prompt']) + ' [SEP] ' + str(row0['B'])
print('Character length:', len(formatted))  # 407

## Q3-Q4: MCQ Tokenization

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize_mcq_row(row, max_length=128):
    prompt = str(row['prompt'])
    ids_list, mask_list = [], []
    for label in LABELS:
        enc = tokenizer(prompt, str(row[label]),
                       max_length=max_length, padding='max_length',
                       truncation=True, return_tensors='pt')
        ids_list.append(enc['input_ids'].squeeze(0))
        mask_list.append(enc['attention_mask'].squeeze(0))
    return torch.stack(ids_list), torch.stack(mask_list)

ids, mask = tokenize_mcq_row(train.iloc[0])
print('Single row shape:', ids.unsqueeze(0).shape)  # [1, 5, 128]
print('Second dimension:', ids.shape[0])  # 5

# Batch of 16
batch_ids = torch.stack([tokenize_mcq_row(train.iloc[i])[0] for i in range(16)])
print('Batch shape:', batch_ids.shape)  # [16, 5, 128]
print('Total token positions:', 16 * 5 * 128)  # 10240

## Q5-Q6: Multiple-Choice Model Outputs

In [ ]:
model = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')

ids, mask = tokenize_mcq_row(train.iloc[0])
with torch.no_grad():
    out = model(input_ids=ids.unsqueeze(0), attention_mask=mask.unsqueeze(0))
print('Logits shape:', out.logits.shape)  # [1, 5]
print('Number of logits:', out.logits.shape[1])  # 5

# With labels
label_tensor = torch.tensor([label2idx[train.iloc[0]['answer']]])
out_loss = model(input_ids=ids.unsqueeze(0), attention_mask=mask.unsqueeze(0), labels=label_tensor)
print('Loss dimensions:', out_loss.loss.dim())  # 0 (scalar)

## Q7: LoRA Trainable Parameters

In [ ]:
base_model = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8, lora_alpha=16,
    target_modules=['query', 'value'],
    lora_dropout=0.1, bias='none'
)
peft_model = get_peft_model(base_model, lora_config)
trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
print('Trainable parameters:', trainable)  # 294912
peft_model.print_trainable_parameters()

## Q8: HuggingFace Dataset Preparation

In [ ]:
class MCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        ids_list, mask_list = [], []
        for l in LABELS:
            enc = self.tokenizer(str(row['prompt']), str(row[l]),
                                max_length=self.max_length, padding='max_length',
                                truncation=True, return_tensors='pt')
            ids_list.append(enc['input_ids'].squeeze(0))
            mask_list.append(enc['attention_mask'].squeeze(0))
        return {'input_ids': torch.stack(ids_list),
                'attention_mask': torch.stack(mask_list),
                'labels': torch.tensor(label2idx[row['answer']])}

ds = MCQDataset(train.head(100), tokenizer)
item = ds[0]
print('input_ids shape:', item['input_ids'].shape)  # [5, 128]
print('Tokenized choices:', item['input_ids'].shape[0])  # 5

## Q9-Q10: Tiny LoRA Fine-Tuning

In [ ]:
tiny_ds = MCQDataset(train.head(32), tokenizer, max_length=64)

tiny_model = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')
tiny_lora = get_peft_model(tiny_model, lora_config)

args = TrainingArguments(
    output_dir='./tmp_m4',
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    remove_unused_columns=False
)
trainer = Trainer(model=tiny_lora, args=args, train_dataset=tiny_ds)
result = trainer.train()
print('Final global_step:', trainer.state.global_step)  # 4

# Inference on row 0
item = tiny_ds[0]
tiny_lora.eval()
with torch.no_grad():
    out = tiny_lora(input_ids=item['input_ids'].unsqueeze(0),
                    attention_mask=item['attention_mask'].unsqueeze(0))
    probs = torch.softmax(out.logits, dim=-1)
    print('Prob of E:', probs[0][4].item())  # ~0.2034